**UNIVERSIDAD DEL QUINDIO** - 2026-2

**PARCIAL #3- ANALISIS DE DATOS**

 **ESTUDIANTE:** Ana Lucia Suaza
 


**EJERCICIO #1**

Para ingresar al sitio web de la empresa, a los clientes se les proporciona un nombre de usuario el cual es una cadena de texto conformada por el customerid asociado a cada usuario, seguido de un guión, las primeras 2 letras de firstname (en mayúsculas) ,otro guión las primeras 2 letras del lastname (en minúsculas), otro guión y termina con la cadena genérica “AW” (mayúsculas) que es la misma para todos los clientes. Esta cadena se denomina user_name.

Desarrollar una consulta SQL para obtener una tabla con el customerid, el fullname y el user_name de la persona.

In [0]:
%sql
USE CATALOG workspace;
CREATE VOLUME IF NOT EXISTS workspace.electiva.datos_final;
USE SCHEMA electiva;

-- 1. Customers
CREATE OR REPLACE TABLE customers AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/customers.csv', format => 'csv', header => true, inferSchema => true);

-- 2. Employee (Nota: el CSV se llama employees.csv, la tabla se pide como employee)
CREATE OR REPLACE TABLE employee AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/employees.csv', format => 'csv', header => true, inferSchema => true);

-- 3. Orders
CREATE OR REPLACE TABLE orders AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/orders.csv', format => 'csv', header => true, inferSchema => true);

-- 4. Product Categories
CREATE OR REPLACE TABLE product_categories AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/productcategories.csv', format => 'csv', header => true, inferSchema => true);

-- 5. Product Sub Categories
CREATE OR REPLACE TABLE product_sub_categories AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/productsubcategories.csv', format => 'csv', header => true, inferSchema => true);

-- 6. Products
CREATE OR REPLACE TABLE products AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/products.csv', format => 'csv', header => true, inferSchema => true);

-- 7. Vendor
CREATE OR REPLACE TABLE vendor AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/vendors.csv', format => 'csv', header => true, inferSchema => true);

-- 8. Vendor Product
CREATE OR REPLACE TABLE vendor_product AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/vendorproduct.csv', format => 'csv', header => true, inferSchema => true);
     

In [0]:
%sql
SELECT 
    customerid,
    fullname,
    CONCAT(
        CAST(customerid AS STRING), '-',
        UPPER(LEFT(firstname, 2)), '-',
        LOWER(LEFT(lastname, 2)), '-AW'
    ) AS user_name
FROM workspace.electiva.customers
ORDER BY customerid ASC;

**EJERCICIO #2**

Dentro de las tablas existentes no existe ninguna tabla que contenga este user_name para ser consultado directamente. Dentro del esquema que usted ha creado ,cree una tabla llamada page_user que contenga dos columnas: el customerid y el user_name e inserte esta información en la tabla recién creada para ser consultada directamente.

**Nota :** Consultar en la documentación de databricks sql los comandos: CREATE TABLE insert into

In [0]:
%sql
-- 1. Crear la estructura de la tabla
CREATE OR REPLACE TABLE workspace.electiva.page_user (
    customerid INT,
    user_name STRING
);

-- 2. Insertar los datos desde la consulta del Punto 1
INSERT INTO workspace.electiva.page_user (customerid, user_name)
SELECT 
    customerid,
    CONCAT(
        CAST(customerid AS STRING), '-',
        UPPER(LEFT(firstname, 2)), '-',
        LOWER(LEFT(lastname, 2)), '-AW'
    ) AS user_name
FROM workspace.electiva.customers;

-- 3. Verificar resultado
SELECT * FROM workspace.electiva.page_user ORDER BY customerid ASC;

**EJERCICIO #3**

Durante el mes de agosto de cada año, se aplica un descuento dependiendo de la categoría del producto de la siguiente manera:

10% a todas las orders dentro de la category_name= ‘Bikes’
15% a todas las orders dentro de la category_name=’Clothing’
8% a todas las orders dentro de la category_name=’Components’
20% a todas las orders dentro de la category_name=’Accessories’ Cree una tabla nueva en el esquema. En esta tabla se deben mostrar todas las orders realizadas en el mes de agosto de 2011, y para cada una de las orders mostrar los campos :

● productname

● modelname

● user_name

● salesorderid

● categoryname

● orderdate

● linetotal

Las columnas anteriormente mencionadas ya vienen en las tablas, además debe crear dos columnas adicionales para la vista :

● Una columna indicando qué porcentaje de descuento se aplica a esta compra por estar en temporada de mitad de año, esta columna se debe llamar descuento_mitad_año y debe contener el descuento correspondiente que aplique en cada fila, si no le aplica ningún descuento a la orden por no pertenecer a una categoría, el descuento es 0

● Una columna llamada total_descuentos_aplicados la cual se calcula restando de la columna linetotal el descuento , es decir linetotal-linetotal*descuento_mitad_año.

**Nota :** consultar comandos de databricks sql create or replace view ilike case when with

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.`v_utl_ordenes_mitad_año` AS
WITH ordenes_agosto AS (
    SELECT 
        p.productname,
        p.modelname,
        pu.user_name,
        o.salesorderid,
        pc.categoryname,
        CAST(o.orderdate AS DATE) AS orderdate,
        o.linetotal,
        CASE 
            WHEN pc.categoryname ILIKE 'Bikes' THEN 0.10
            WHEN pc.categoryname ILIKE 'Clothing' THEN 0.15
            WHEN pc.categoryname ILIKE 'Components' THEN 0.08
            WHEN pc.categoryname ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
    FROM workspace.electiva.orders o
    INNER JOIN workspace.electiva.page_user pu 
        ON o.customerid = pu.customerid
    INNER JOIN workspace.electiva.products p 
        ON o.productid = p.productid
    LEFT JOIN workspace.electiva.product_sub_categories psc 
        ON p.SubCategoryID = psc.SubCategoryID
    LEFT JOIN workspace.electiva.product_categories pc 
        ON psc.CategoryID = pc.CategoryID
    WHERE YEAR(o.orderdate) = 2011 
      AND MONTH(o.orderdate) = 8
)
SELECT 
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    `descuento_mitad_año`,
    (linetotal - (linetotal * `descuento_mitad_año`)) AS total_descuentos_aplicados
FROM ordenes_agosto;

-- Verificar resultado
SELECT * FROM workspace.electiva.`v_utl_ordenes_mitad_año`;

**EJERCICIO #4**

Su empresa quiere desplegar campañas publicitarias y de segmentación de clientes para incrementar las ventas del año 2011 y para ello usted debe crear un pequeño sistema de clasificación de clientes que cumpla los siguientes requerimientos:

- Los clientes que hayan hecho 3 órdenes en el año son considerados clientes Oro.
- Los clientes que hayan hecho 2 órdenes en el año son considerados clientes Plata.
- Los clientes que hayan hecho menos de 2 órdenes al año son clientes Ocasional


La cantidad de órdenes que un cliente hizo en el año 2011 se obtiene contando la cantidad de salesorderid distintos, cada salesorderid es el identificador de una orden hecha por un
cliente, queremos identificar la cantidad de salesorderid distintos para cada cliente, para así poder clasificarlos en alguna de las categorías.

Crear una vista materializada en el esquema, llamada customer_class que contenga el user_name de todos los clientes del año 2011 , una columna user_level donde indique si el cliente es “Oro”, “Plata” o “Ocasional”, además debe contener para cada cliente su customerid, su customer fullname y la cantidad de órdenes que realizó en el año

**Nota:** consultar comando create materialized view.

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
WITH conteo_ordenes AS (
    SELECT 
        o.customerid,
        COUNT(DISTINCT o.salesorderid) AS ammount_of_orders
    FROM workspace.electiva.orders o
    WHERE YEAR(o.orderdate) = 2011
    GROUP BY o.customerid
)
SELECT 
    co.ammount_of_orders,
    pu.user_name,
    c.fullname AS customerfullname,
    CASE 
        WHEN co.ammount_of_orders >= 3 THEN 'Oro'
        WHEN co.ammount_of_orders = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,
    co.customerid
FROM conteo_ordenes co
INNER JOIN workspace.electiva.customers c 
    ON co.customerid = c.customerid
INNER JOIN workspace.electiva.page_user pu 
    ON co.customerid = pu.customerid;

-- Verificar resultado
SELECT * FROM workspace.electiva.customer_class;
     